<a href="https://colab.research.google.com/github/datacuriouss/fashion-retail-discount-analysis/blob/main/FashionRetailAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fashion Retail Discount Analysis

## Project Overview
This project analyses transactional and discount data from a global fashion retailer
operating across 7 countries and 35 stores over a 2-year period.

## Business Question
Does discounting actually drive more sales — or has constant discounting
trained customers to buy regardless of whether a discount exists?

This is a question I observed firsthand working in retail at Strandbags,
where promotional sales were near-constant. I wanted to test with real data
whether discounts genuinely move the needle on sales volume and revenue.

## Approach
- Python (pandas, numpy) — data loading, cleaning & feature engineering
- SQL (sqlite3) — business questions & aggregations
- Tableau — interactive dashboard

## Data Source
Global Fashion Retail Sales dataset — synthetic transactional data simulating
2 years of sales across 35 stores in 7 countries.
Source: Kaggle (ricgomes/global-fashion-retail-stores-dataset)

## 1. Loading the Data

Loading all 6 files from the dataset — transactions, discounts, products,
stores, employees and customers. I'll only use what's relevant to the
discount analysis, but good to have everything available.

In [2]:
import pandas as pd
import numpy as np
import io
from google.colab import files

uploaded = files.upload()

Saving customers.csv to customers.csv
Saving discounts.csv to discounts.csv
Saving employees.csv to employees.csv
Saving products.csv to products.csv
Saving stores.csv to stores.csv
Saving transactions.csv to transactions.csv


In [3]:
transactions = pd.read_csv(io.BytesIO(uploaded['transactions.csv']))
discounts = pd.read_csv(io.BytesIO(uploaded['discounts.csv']))
products = pd.read_csv(io.BytesIO(uploaded['products.csv']))
stores = pd.read_csv(io.BytesIO(uploaded['stores.csv']))
employees = pd.read_csv(io.BytesIO(uploaded['employees.csv']))
customers = pd.read_csv(io.BytesIO(uploaded['customers.csv']))

print("transactions:", transactions.shape)
print("discounts:", discounts.shape)
print("products:", products.shape)
print("stores:", stores.shape)
print("employees:", employees.shape)
print("customers:", customers.shape)

transactions: (6416827, 19)
discounts: (181, 6)
products: (17940, 12)
stores: (35, 8)
employees: (404, 4)
customers: (1643306, 9)


/tmp/ipykernel_247/931464052.py:6: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  customers = pd.read_csv(io.BytesIO(uploaded['customers.csv']))


## 2. Initial Exploration

Before cleaning anything, I want to understand what each key file
looks like — column names, data types, and missing values.

In [4]:
print("=== TRANSACTIONS ===")
print(transactions.dtypes)
print("\nMissing values:")
print(transactions.isna().sum())

=== TRANSACTIONS ===
Invoice ID           object
Line                  int64
Customer ID           int64
Product ID            int64
Size                 object
Color                object
Unit Price          float64
Quantity              int64
Date                 object
Discount            float64
Line Total          float64
Store ID              int64
Employee ID           int64
Currency             object
Currency Symbol      object
SKU                  object
Transaction Type     object
Payment Method       object
Invoice Total       float64
dtype: object

Missing values:
Invoice ID                0
Line                      0
Customer ID               0
Product ID                0
Size                 413102
Color               4350783
Unit Price                0
Quantity                  0
Date                      0
Discount                  0
Line Total                0
Store ID                  0
Employee ID               0
Currency                  0
Currency Symbol         

In [6]:
print(transactions.head(3))
print("\nDiscount unique values sample:")
print(transactions['Discount'].value_counts().head(10))
print("\nTransaction Type values:")
print(transactions['Transaction Type'].value_counts())

            Invoice ID  Line  Customer ID  Product ID Size    Color  \
0  INV-US-001-03558761     1        47162         485    M      NaN   
1  INV-US-001-03558761     2        47162        2779    G      NaN   
2  INV-US-001-03558761     3        47162          64    M  NEUTRAL   

   Unit Price  Quantity                 Date  Discount  Line Total  Store ID  \
0        80.5         1  2023-01-01 15:42:00       0.0        80.5         1   
1        31.5         1  2023-01-01 15:42:00       0.4        18.9         1   
2        45.5         1  2023-01-01 15:42:00       0.4        27.3         1   

   Employee ID Currency Currency Symbol               SKU Transaction Type  \
0            7      USD               $        MASU485-M-             Sale   
1            7      USD               $       CHCO2779-G-             Sale   
2            7      USD               $  MACO64-M-NEUTRAL             Sale   

  Payment Method  Invoice Total  
0           Cash          126.7  
1           C

## 3. Data Cleaning

Key issues to fix:
- Filter out returns, keep only Sales
- Fix Date column from text to datetime
- Drop Color column (68% missing, not needed for analysis)
- Rename 'Discont' column in discounts file (typo)
- Standardise column names (remove spaces)

In [8]:
transactions = transactions[transactions['Transaction Type'] == 'Sale']
print("Rows after removing returns:", transactions.shape[0])

Rows after removing returns: 6077200


In [9]:
# Remove spaces from column names - makes them easier to work with
transactions.columns = transactions.columns.str.replace(' ', '_')
discounts.columns = discounts.columns.str.replace(' ', '_')
products.columns = products.columns.str.replace(' ', '_')
stores.columns = stores.columns.str.replace(' ', '_')

print(transactions.columns.tolist())

['Invoice_ID', 'Line', 'Customer_ID', 'Product_ID', 'Size', 'Color', 'Unit_Price', 'Quantity', 'Date', 'Discount', 'Line_Total', 'Store_ID', 'Employee_ID', 'Currency', 'Currency_Symbol', 'SKU', 'Transaction_Type', 'Payment_Method', 'Invoice_Total']


In [10]:
# Fix typo in discounts column name
discounts = discounts.rename(columns={'Discont': 'Discount'})
print(discounts.columns.tolist())

['Start', 'End', 'Discount', 'Description', 'Category', 'Sub_Category']


Converting Date from text to datetime so we can do time-based analysis
like monthly sales trends.

In [11]:
# Fix Date column - currently stored as text, needs to be datetime
transactions['Date'] = pd.to_datetime(transactions['Date'])
print(transactions['Date'].dtype)
print(transactions['Date'].head(3))

datetime64[ns]
0   2023-01-01 15:42:00
1   2023-01-01 15:42:00
2   2023-01-01 15:42:00
Name: Date, dtype: datetime64[ns]


Dropping Color column — 68% of values are missing and color is not
relevant to our discount effectiveness analysis.

In [14]:
# Drop Color - too many missing values and not needed for our analysis
transactions = transactions.drop(columns=['Color'])
print("Columns remaining:", transactions.columns.tolist())

KeyError: "['Color'] not found in axis"

In [16]:
print("Columns remaining:", transactions.columns.tolist())


Columns remaining: ['Invoice_ID', 'Line', 'Customer_ID', 'Product_ID', 'Size', 'Unit_Price', 'Quantity', 'Date', 'Discount', 'Line_Total', 'Store_ID', 'Employee_ID', 'Currency', 'Currency_Symbol', 'SKU', 'Transaction_Type', 'Payment_Method', 'Invoice_Total']


In [17]:
# Dropping columns not needed for discount analysis
transactions = transactions.drop(columns=['Currency_Symbol', 'SKU', 'Line', 'Employee_ID', 'Size'])
print("Columns remaining:", transactions.columns.tolist())

Columns remaining: ['Invoice_ID', 'Customer_ID', 'Product_ID', 'Unit_Price', 'Quantity', 'Date', 'Discount', 'Line_Total', 'Store_ID', 'Currency', 'Transaction_Type', 'Payment_Method', 'Invoice_Total']


In [18]:
print(transactions.isna().sum())

Invoice_ID          0
Customer_ID         0
Product_ID          0
Unit_Price          0
Quantity            0
Date                0
Discount            0
Line_Total          0
Store_ID            0
Currency            0
Transaction_Type    0
Payment_Method      0
Invoice_Total       0
dtype: int64


In [20]:
print(transactions[['Unit_Price', 'Quantity', 'Discount', 'Line_Total', 'Invoice_Total']].describe().round(2))

       Unit_Price   Quantity    Discount  Line_Total  Invoice_Total
count  6077200.00  6077200.0  6077200.00  6077200.00     6077200.00
mean       132.48        1.1        0.13      127.70         274.36
std        185.12        0.4        0.20      203.72         519.09
min          2.00        1.0        0.00        1.40           1.40
25%         32.50        1.0        0.00       27.45          39.00
50%         51.00        1.0        0.00       46.50          92.25
75%        116.50        1.0        0.25      120.75         256.55
max       1153.50        3.0        0.60     3460.50        8977.00


## 4. Feature Engineering

Creating new columns to support the discount analysis:
- Discount_Percentage: converting discount from decimal to percentage (easier to read)
- Discounted: a simple yes/no flag for whether a discount was applied
- Month and Year: extracted from Date for time-based trend analysis

In [21]:
# Convert discount from decimal to percentage
transactions['Discount_Percentage'] = (transactions['Discount'] * 100).round(0).astype(int)

# Flag whether a discount was applied
transactions['Discounted'] = transactions['Discount'] > 0

# Extract month and year for trend analysis
transactions['Year'] = transactions['Date'].dt.year
transactions['Month'] = transactions['Date'].dt.month

print(transactions[['Discount', 'Discount_Percentage', 'Discounted', 'Year', 'Month']].head(5))

   Discount  Discount_Percentage  Discounted  Year  Month
0       0.0                    0       False  2023      1
1       0.4                   40        True  2023      1
2       0.4                   40        True  2023      1
3       0.4                   40        True  2023      1
4       0.0                    0       False  2023      1


## 5. Cleaning Supporting Files

Cleaning discounts, products and stores before exporting everything
for SQL analysis.

In [24]:
# --- DISCOUNTS ---
# Convert Start and End from text to datetime
discounts['Start'] = pd.to_datetime(discounts['Start'])
discounts['End'] = pd.to_datetime(discounts['End'])

# Fill 10 missing Category and Sub_Category values with 'Unknown'
discounts['Category'] = discounts['Category'].fillna('Unknown')
discounts['Sub_Category'] = discounts['Sub_Category'].fillna('Unknown')

print("Discounts cleaned:")
print(discounts.dtypes)
print(discounts.isna().sum())

Discounts cleaned:
Start           datetime64[ns]
End             datetime64[ns]
Discount               float64
Description             object
Category                object
Sub_Category            object
dtype: object
Start           0
End             0
Discount        0
Description     0
Category        0
Sub_Category    0
dtype: int64


In [25]:
# --- PRODUCTS ---
# Only keep English description, drop other languages
# Drop Color and Sizes - not needed for discount analysis
products = products.drop(columns=[
    'Description_PT', 'Description_DE', 'Description_FR',
    'Description_ES', 'Description_ZH', 'Color', 'Sizes'
])

print("Products cleaned:")
print(products.columns.tolist())
print(products.isna().sum())

Products cleaned:
['Product_ID', 'Category', 'Sub_Category', 'Description_EN', 'Production_Cost']
Product_ID         0
Category           0
Sub_Category       0
Description_EN     0
Production_Cost    0
dtype: int64


## 6. Exporting Clean Data

Exporting all cleaned files for SQL analysis and Tableau visualisation.

In [26]:
# Export all cleaned files
transactions.to_csv('transactions_cleaned.csv', index=False)
discounts.to_csv('discounts_cleaned.csv', index=False)
products.to_csv('products_cleaned.csv', index=False)
stores.to_csv('stores_cleaned.csv', index=False)

files.download('transactions_cleaned.csv')
files.download('discounts_cleaned.csv')
files.download('products_cleaned.csv')
files.download('stores_cleaned.csv')

print("✅ All files exported!")
print("transactions:", transactions.shape)
print("discounts:", discounts.shape)
print("products:", products.shape)
print("stores:", stores.shape)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ All files exported!
transactions: (6077200, 17)
discounts: (181, 6)
products: (17940, 5)
stores: (35, 8)
